# CampaignOptima: Phase 2 — Predictive Modeling & Bayesian Hyperparameter Optimization

**Objective:** Benchmarking candidate classifiers (Logistic Regression, Random Forest, LightGBM, XGBoost) and running Bayesian hyperparameter tuning using **Optuna** explicitly targeting **Precision-Recall AUC (PR-AUC)** rather than uninformative accuracy.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, roc_curve, auc

ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.data_loader import BankMarketingDataLoader
from src.features import CampaignOptimaFeaturePipeline
from src.models import ModelBenchmarkEngine, evaluate_classifier

## 1. Feature Preprocessing & SMOTE Balancing
We fit the pipeline on training data and oversample minority conversion events using SMOTE.

In [ ]:
loader = BankMarketingDataLoader()
df_raw = loader.load_raw_data()
X, y = loader.clean_and_prepare(df_raw)
X_train, X_test, y_train, y_test = loader.get_stratified_train_test_split(X, y)

pipeline = CampaignOptimaFeaturePipeline()
X_train_proc = pipeline.fit_transform(X_train)
X_test_proc = pipeline.transform(X_test)

X_train_res, y_train_res = pipeline.balance_with_smote(X_train_proc, y_train)
print(f"Post-SMOTE Training Matrix Shape: {X_train_res.shape}")

## 2. Multi-Model Cross-Validation (PR-AUC Benchmark)

In [ ]:
benchmark = ModelBenchmarkEngine()
cv_results = benchmark.run_cross_validation(X_train_res, y_train_res, cv_folds=5)

cv_df = pd.DataFrame(cv_results).T
cv_df.sort_values(by="mean_pr_auc", ascending=False)

## 3. Optuna Bayesian Optimization for XGBoost
Tuning `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`, and `scale_pos_weight` targeting PR-AUC.

In [ ]:
tuned_xgb = benchmark.tune_xgboost_with_optuna(X_train_res, y_train_res, n_trials=20)
metrics_tuned = evaluate_classifier(tuned_xgb, X_test_proc, y_test)
print("Tuned XGBoost Test Performance:", metrics_tuned)

## 4. Precision-Recall Curves Across Models

In [ ]:
plt.figure(figsize=(8, 6))
y_probs = tuned_xgb.predict_proba(X_test_proc)[:, 1]
prec, rec, _ = precision_recall_curve(y_test, y_probs)
pr_auc = auc(rec, prec)

plt.plot(rec, prec, label=f"Tuned XGBoost (PR-AUC = {pr_auc:.3f})", color="#2563EB", lw=2.5)
plt.axhline(y=y_test.mean(), color="grey", linestyle="--", label=f"Random Baseline ({y_test.mean():.2%})")
plt.title("Precision-Recall Curve (Customer Propensity)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.show()